In [ ]:
#V1
import os
import torch
import torch_npu
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
import matplotlib.pyplot as plt
from transformers.trainer_callback import TrainerCallback
import warnings
warnings.filterwarnings("ignore")
# 设置多NPU环境
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12355'

# 自动检测可用的NPU数量
device = torch.device('npu' if torch_npu.npu.is_available() else "cpu")
npu_count = torch_npu.npu.device_count()
print(f"检测到 {npu_count} 张NPU卡")

device = torch.device('npu' if torch_npu.npu.is_available() else "cpu")

base_path = "/opt/huawei/edu-apaas/src/init/luoyuping/pangu1B_training"
model_path = os.path.join(base_path,"openPangu-Embedded-1B-V1.1")
output_path = os.path.join(base_path, "outputs")
data_path = os.path.join(base_path,"data")
tokenizer = AutoTokenizer.from_pretrained(model_path,trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_path,trust_remote_code=True)


ds = load_dataset("parquet", data_files={
    "train": os.path.join(data_path, "train-00000-of-00001-a09b74b3ef9c3b56.parquet")
})

test_size = 100
ds["train"] = ds["train"].select(range(min(test_size, len(ds["train"]))))
print(f"使用 {len(ds['train'])} 个样本进行测试训练")

def preprocess_function(examples):
    # 正确的方式：对每个样本分别处理
    inputs = []
    targets = []
    
    for i in range(len(examples["instruction"])):
        # 处理每个样本
        instruction = examples["instruction"][i] if i < len(examples["instruction"]) else ""
        input_text = examples["input"][i] if i < len(examples["input"]) else ""
        output_text = examples["output"][i] if i < len(examples["output"]) else ""
        
        # 构建输入和目标
        input_str = f"{instruction}\n{input_text}".strip()
        target_str = output_text
        
        inputs.append(input_str)
        targets.append(target_str)
    
    # 对批量数据进行tokenize
    model_inputs = tokenizer(inputs, truncation=True, padding="max_length", max_length=1024)
    labels = tokenizer(targets, truncation=True, padding="max_length", max_length=1024)["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs
tokenized_datasets = ds.map(preprocess_function, batched=True)


training_args = TrainingArguments(
    output_dir=output_path,
    overwrite_output_dir=True,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    bf16=True,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_steps=100,
    save_total_limit=2,
    logging_dir=os.path.join(output_path, "logs"),
    logging_steps=10,
    #evaluation_strategy="steps",
    eval_steps=50,
    dataloader_drop_last=True,
    dataloader_num_workers=8,
    gradient_accumulation_steps=4,
    local_rank=int(os.getenv('LOCAL_RANK', -1)),
     torch_compile=False,
     remove_unused_columns=True,
    label_names=["labels"],
)

# 定义 Trainer
print("初始化 Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = tokenized_datasets["train"],  # 训练集
    eval_dataset = tokenized_datasets["train"],   # 评估集（这里简单用训练集代替）
)

# 开始训练
print("开始训练...")
trainer.train()

# 保存最终模型
print("保存模型...")
if trainer.args.local_rank in [-1, 0]:
    model.save_pretrained(os.path.join(output_path, "final_model"))
    tokenizer.save_pretrained(os.path.join(output_path, "final_model"))

print("训练完成！")


检测到 8 张NPU卡
使用 100 个样本进行测试训练
初始化 Trainer...
开始训练...


Step,Training Loss
10,17.052200
20,15.885500
30,9.480200


保存模型...


In [9]:
#v2
import csv
import os
import torch
import torch_npu
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore")
# 设置多NPU环境
#os.environ['MASTER_ADDR'] = 'localhost'
#os.environ['MASTER_PORT'] = '12355'
class CustomTrainer(Trainer):
    def __init__(self,*args,log_file = 'training_log.csv',**kwargs):
        super().__init__(*args,**kwargs)
        self.log_file = log_file
        with open(self.log_file, mode='w',newline="") as file:
            writer = csv.writer(file)
            writer.writerow(['step','loss','learning_rate','epoch'])
    def log(self, logs):
        super().log(logs)
        step = self.state.global_step

        with open(self.log_file,mode='a',newline='') as file:
            writer = csv.writer(file)
            writer.writerow([
                step,
                logs.get('loss','N/A'),
                logs.get('learning_rate',"N/A"),
                logs.get('epoch','N/A')
            ])


base_path = "/opt/huawei/edu-apaas/src/init/luoyuping/pangu1B_training"
model_path = os.path.join(base_path, "openPangu-Embedded-1B-V1.1")
output_path = os.path.join(base_path, "outputs")
data_path = os.path.join(base_path, "data")
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True)

ds = load_dataset("parquet", data_files={
    "train": os.path.join(data_path, "train-00000-of-00001-a09b74b3ef9c3b56.parquet")
})

test_size = 100
ds["train"] = ds["train"].select(range(min(test_size, len(ds["train"]))))
print(f"使用 {len(ds['train'])} 个样本进行测试训练")


def preprocess_function(examples):
    # 正确的方式：对每个样本分别处理
    inputs = []
    targets = []

    for i in range(len(examples["instruction"])):
        # 处理每个样本
        instruction = examples["instruction"][i] if i < len(examples["instruction"]) else ""
        input_text = examples["input"][i] if i < len(examples["input"]) else ""
        output_text = examples["output"][i] if i < len(examples["output"]) else ""

        # 构建输入和目标
        input_str = f"{instruction}\n{input_text}".strip()
        target_str = output_text

        inputs.append(input_str)
        targets.append(target_str)

    # 对批量数据进行tokenize
    model_inputs = tokenizer(inputs, truncation=True, padding="max_length", max_length=4096)
    labels = tokenizer(targets, truncation=True, padding="max_length", max_length=4096)["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs


tokenized_datasets = ds.map(preprocess_function, batched=True)

training_args = TrainingArguments(
    output_dir=output_path,
    overwrite_output_dir=True,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    bf16=True,
    dataloader_drop_last=True,
    save_total_limit=2,
    save_steps=100,
    logging_dir=os.path.join(output_path, "logs"),
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    logging_steps=10,
    dataloader_num_workers=8,
    local_rank=int(os.getenv('LOCAL_RANK', -1)),#单设备
     remove_unused_columns=True,
     #torch_compile=False,
    label_names=["labels"],
    # evaluation_strategy="steps",
    #eval_steps=50,
)

# 定义 Trainer
print("初始化 Trainer...")
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],  # 训练集
    eval_dataset=tokenized_datasets["train"],  # 评估集（这里简单用训练集代替）
    log_file = 'training_log.csv'
)

# 开始训练
print("开始训练...")
trainer.train()

# 保存最终模型
print("保存模型...")
if trainer.args.local_rank in [-1, 0]:
    model.save_pretrained(os.path.join(output_path, "final_model"))
    tokenizer.save_pretrained(os.path.join(output_path, "final_model"))

print("训练完成！")


使用 100 个样本进行测试训练
初始化 Trainer...
开始训练...


BackendCompilerFailed: backend='inductor' raised:
TypeError: 'NoneType' object is not callable

Set TORCH_LOGS="+dynamo" and TORCHDYNAMO_VERBOSE=1 for more information


You can suppress this exception and fall back to eager by setting:
    import torch._dynamo
    torch._dynamo.config.suppress_errors = True


In [1]:
import os
import torch
import torch_npu
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
import matplotlib.pyplot as plt
from transformers.trainer_callback import TrainerCallback

# 设置多NPU环境
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12355'

# 自动检测可用的NPU数量
device = torch.device('npu' if torch_npu.npu.is_available() else "cpu")
npu_count = torch_npu.npu.device_count()
print(f"检测到 {npu_count} 张NPU卡")

/home/service/.local/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/service/.local/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC3/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


检测到 8 张NPU卡


In [1]:
pip install --upgrade accelerate -i https://repo.huaweicloud.com/repository/pypi/simple/ --trusted-host repo.huaweicloud.com

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://repo.huaweicloud.com/repository/pypi/simple/
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.26.1
    Uninstalling accelerate-0.26.1:
      Successfully uninstalled accelerate-0.26.1
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch

# 检查 NPU 是否可用
if hasattr(torch, 'npu') and torch.npu.is_available():
    print("NPU 可用！")
else:
    print("NPU 不可用，检查硬件或环境配置。")


In [4]:
ls -ld /usr/local/Ascend/ascend-toolkit


drwxr-xr-x 4 root root 4096 Dec  4  2024 /usr/local/Ascend/ascend-toolkit/


In [2]:
pip install transformers==4.53.2 -i https://repo.huaweicloud.com/repository/pypi/simple/ --trusted-host repo.huaweicloud.com

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://repo.huaweicloud.com/repository/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 78.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 64.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 19.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 18.5 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.37.1
    Uninstalling transformers-4.37.1:
      Successfully uninstalled transformers-4.37.1
  Consider adding this directory to PATH or, if you prefer to s

In [8]:
python -m pip install --upgrade pip -i https://repo.huaweicloud.com/repository/pypi/simple/ --trusted-host repo.huaweicloud.com

SyntaxError: invalid syntax (2948083916.py, line 1)

In [9]:
pip install --upgrade pip -i https://repo.huaweicloud.com/repository/pypi/simple/ --trusted-host repo.huaweicloud.com

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://repo.huaweicloud.com/repository/pypi/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.7 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip show transformers


Name: transformers
Version: 4.53.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /home/service/.local/lib/python3.10/site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: lm_eval, peft, trl
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip list


In [4]:
#V1
import os
import torch
torch.npu.empty_cache()
import torch_npu
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import load_dataset
import matplotlib.pyplot as plt
from transformers.trainer_callback import TrainerCallback
import warnings
warnings.filterwarnings("ignore")

base_path = "/opt/huawei/edu-apaas/src/init/luoyuping/pangu1B_training"
model_path = os.path.join(base_path,"openPangu-Embedded-1B-V1.1")
output_path = os.path.join(base_path, "outputs")
data_path = os.path.join(base_path,"data")
tokenizer = AutoTokenizer.from_pretrained(model_path,trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_path,trust_remote_code=True)


ds = load_dataset("parquet", data_files={
    "train": os.path.join(data_path, "train-00000-of-00001-a09b74b3ef9c3b56.parquet")
})

test_size = 100
ds["train"] = ds["train"].select(range(min(test_size, len(ds["train"]))))
print(f"使用 {len(ds['train'])} 个样本进行测试训练")

def preprocess_function(examples):
    # 正确的方式：对每个样本分别处理
    inputs = []
    targets = []
    
    for i in range(len(examples["instruction"])):
        # 处理每个样本
        instruction = examples["instruction"][i] if i < len(examples["instruction"]) else ""
        input_text = examples["input"][i] if i < len(examples["input"]) else ""
        output_text = examples["output"][i] if i < len(examples["output"]) else ""
        
        # 构建输入和目标
        input_str = f"{instruction}\n{input_text}".strip()
        target_str = output_text
        
        inputs.append(input_str)
        targets.append(target_str)
    
    # 对批量数据进行tokenize
    model_inputs = tokenizer(inputs, truncation=True, padding="max_length", max_length=1024)
    labels = tokenizer(targets, truncation=True, padding="max_length", max_length=1024)["input_ids"]
    model_inputs["labels"] = labels
    return model_inputs
tokenized_datasets = ds.map(preprocess_function, batched=True)


training_args = TrainingArguments(
    output_dir=output_path,
    overwrite_output_dir=True,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    bf16=True,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    save_steps=100,
    save_total_limit=2,
    logging_dir=os.path.join(output_path, "logs"),
    logging_steps=10,
    #evaluation_strategy="steps",
    eval_steps=50,
    dataloader_drop_last=True,
    dataloader_num_workers=8,
    gradient_accumulation_steps=4,
    local_rank=int(os.getenv('LOCAL_RANK', -1)),
     torch_compile=False,
     remove_unused_columns=True,
    label_names=["labels"],
)

# 定义 Trainer
print("初始化 Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = tokenized_datasets["train"],  # 训练集
    eval_dataset = tokenized_datasets["train"],   # 评估集（这里简单用训练集代替）
)

# 开始训练
print("开始训练...")
trainer.train()

# 保存最终模型
print("保存模型...")
if trainer.args.local_rank in [-1, 0]:
    model.save_pretrained(os.path.join(output_path, "final_model"))
    tokenizer.save_pretrained(os.path.join(output_path, "final_model"))

print("训练完成！")


AttributeError: module 'torch' has no attribute 'npu'

In [7]:
pip install torch_npu torchvision_npu -i https://repo.huaweicloud.com/repository/pypi/simple/ --trusted-host repo.huaweicloud.com


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://repo.huaweicloud.com/repository/pypi/simple/
ERROR: Could not find a version that satisfies the requirement torchvision_npu (from versions: none)
ERROR: No matching distribution found for torchvision_npu
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install --upgrade pip 

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.huaweicloud.com/repository/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.5 MB/s eta 0:00:00a 0:00:01
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

names = [
    "ASCEND_VISIBLE_DEVICES",
    "CUDA_VISIBLE_DEVICES",
    "NPU_VISIBLE_DEVICES",   # 有些环境也可能用这种名字
]

print("=== 环境变量检查 ===")
for name in names:
    value = os.environ.get(name)
    print(f"{name} = {value}")


=== 环境变量检查 ===
ASCEND_VISIBLE_DEVICES = 6,7,0,1,2,3,4,5
CUDA_VISIBLE_DEVICES = None
NPU_VISIBLE_DEVICES = 6,7,0,1,2,3,4,5
